In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

e:\studies_master\MlShared\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

models = {
    "small_model": "bigscience/bloom-1b7",   
    "large_model": "bigscience/bloom-7b1"   
}

llms = {}
for name, model_id in models.items():
    print(f"Loading {name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
    llms[name] = {"model": model, "tokenizer": tokenizer}

Using device: cuda
Loading small_model...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
e:\studies_master\MlShared\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\matkan1\.cache\huggingface\hub\models--bigscience--bloom-1b7. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https

Loading large_model...


e:\studies_master\MlShared\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\matkan1\.cache\huggingface\hub\models--bigscience--bloom-7b1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download.

In [ ]:
zero_shot_prompt = "Classify the sentiment of this sentence: 'I love this movie!'. Answer 'Positive' or 'Negative'."
few_shot_prompt = (
    "Classify the sentiment of the following sentences:\n"
    "Example 1: 'The food was amazing.' -> Positive\n"
    "Example 2: 'I hated the traffic today.' -> Negative\n"
    "Example 3: 'The book was fantastic.' -> Positive\n"
    "Now classify this sentence: 'The movie was boring.' ->"
)
generation_prompt = "Question: If a train leaves station A at 60 km/h and another train leaves station B at 80 km/h, when will they meet if the distance is 140 km? Answer step by step:"


gen_params = {
    "max_new_tokens": 100,
    "temperature": 0.7,
    "top_p": 0.9,
    "do_sample": True,
    "pad_token_id": 0
}

def generate_text(model, tokenizer, prompt, num_runs=3, **kwargs):
    outputs = []
    for _ in range(num_runs):
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        output_ids = model.generate(**inputs, **kwargs)
        text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        outputs.append(text)
    return outputs


In [5]:
from IPython.display import display, Markdown

results = {}

for name, data in llms.items():
    display(Markdown(f"## Running experiments for {name}"))
    model = data["model"]
    tokenizer = data["tokenizer"]
    
    gen_params["pad_token_id"] = tokenizer.eos_token_id
    results[name] = {}
    
    # Zero-shot
    zs_outputs = generate_text(model, tokenizer, zero_shot_prompt, **gen_params)
    results[name]["zero_shot"] = zs_outputs
    display(Markdown("### Zero-shot outputs"))
    zs_text = "\n\n".join([f"**{i+1}:** {out}" for i, out in enumerate(zs_outputs)])
    display(Markdown(zs_text))
    
    # Few-shot
    fs_outputs = generate_text(model, tokenizer, few_shot_prompt, **gen_params)
    results[name]["few_shot"] = fs_outputs
    display(Markdown("### Few-shot outputs"))
    fs_text = "\n\n".join([f"**{i+1}:** {out}" for i, out in enumerate(fs_outputs)])
    display(Markdown(fs_text))
    
    # Generative reasoning
    gen_outputs = generate_text(model, tokenizer, generation_prompt, **gen_params)
    results[name]["generation"] = gen_outputs
    display(Markdown("### Generative outputs"))
    gen_text = "\n\n".join([f"**{i+1}:** {out}" for i, out in enumerate(gen_outputs)])
    display(Markdown(gen_text))


## Running experiments for small_model

### Zero-shot outputs

**1:** Classify the sentiment of this sentence: 'I love this movie!'. Answer 'Positive' or 'Negative'. For example, for the sentence 'I love this movie!', the sentiment of the sentence is Positive. The sentiment is determined by a sentiment classifier which is trained on the entire corpus. The sentiment classifier is trained using the J48 decision tree classifier which is a popular sentiment classifier. The sentiment classifier is trained on the entire corpus. The sentiment classifier is trained using the J48 decision tree classifier which is a popular sentiment classifier. The sentiment classifier is trained on the entire corpus. The sentiment classifier is trained

**2:** Classify the sentiment of this sentence: 'I love this movie!'. Answer 'Positive' or 'Negative'. The best way to classify the sentiment of a sentence is to check out the sentiment tags provided by the sentiment tagger. For example, the tagger can provide a positive sentiment tag, for example, 'Positive', and a negative sentiment tag, for example, 'Negative'. The positive sentiment tag indicates that the sentiment of the sentence is positive, while the negative sentiment tag indicates that the sentiment of the sentence is negative. In this paper, we used the Stanford CoreNLP toolkit

**3:** Classify the sentiment of this sentence: 'I love this movie!'. Answer 'Positive' or 'Negative'.

### Few-shot outputs

**1:** Classify the sentiment of the following sentences:
Example 1: 'The food was amazing.' -> Positive
Example 2: 'I hated the traffic today.' -> Negative
Example 3: 'The book was fantastic.' -> Positive
Now classify this sentence: 'The movie was boring.' -> Negative

A:

I think you can use the following regex:
(?<=(?!\W)(?!\W)(?!\W)(?!\W)(?!\W)(?!\W)(?!\W))\W)

This pattern is a bit too complex, but it should get the job done. Note that the pattern is a bit more complex than I originally intended. I removed some of the unnecessary patterns, and I added a space before the last \W, since this is

**2:** Classify the sentiment of the following sentences:
Example 1: 'The food was amazing.' -> Positive
Example 2: 'I hated the traffic today.' -> Negative
Example 3: 'The book was fantastic.' -> Positive
Now classify this sentence: 'The movie was boring.' -> Negative
This can be done by using a supervised learning method, such as Naive Bayes (NB), Support Vector Machines (SVM), K-Nearest Neighbors (KNN) and so on. The algorithm takes a training set and outputs a classifier that predicts the class of the test set. The classifiers are trained on a set of training sentences. Then the algorithm outputs a classifier for a test set. The trained classifier is a good predictor for the new test set

**3:** Classify the sentiment of the following sentences:
Example 1: 'The food was amazing.' -> Positive
Example 2: 'I hated the traffic today.' -> Negative
Example 3: 'The book was fantastic.' -> Positive
Now classify this sentence: 'The movie was boring.' -> Negative
In the previous examples, the sentiment of the sentences is classified as positive, negative or neutral. In the following examples, the sentiment of the sentences is classified as positive, negative, and neutral:
Example 4: 'The music was amazing.' -> Positive
Example 5: 'I hated the traffic today.' -> Negative
Example 6: 'The book was fantastic.' -> Positive
In the previous examples, the sentiment of the sentences is classified as positive, negative, and neutral. In the following

...

**1:** Classify the sentiment of this sentence: 'I love this movie!'. Answer 'Positive' or 'Negative'. The output is shown in Figure 1 .
In the next step, we apply the same approach on the training data, and then we calculate the accuracy. The accuracy of the model is shown in Table 2 . We can see that the accuracy of the model is about 0.87. In the next step, we try to improve the accuracy by using the data augmentation. We use the following data augmentation approach:
• Rotation: We rotate the sentence with a random angle between 0 and 360 degrees.
•

**2:** Classify the sentiment of this sentence: 'I love this movie!'. Answer 'Positive' or 'Negative'.", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "",

**3:** Classify the sentiment of this sentence: 'I love this movie!'. Answer 'Positive' or 'Negative'. This is a simple example of a sentiment analysis task. The task is to predict whether the sentiment of the sentence is positive or negative. There are two main approaches to this problem. The first is to build a classifier that can determine the polarity of the sentence given a fixed set of features. The second approach is to build a model that learns the polarity of the sentence from data. In this work, we use the second approach.
The main challenge of this task is to identify the most

### Few-shot outputs

**1:** Classify the sentiment of the following sentences:
Example 1: 'The food was amazing.' -> Positive
Example 2: 'I hated the traffic today.' -> Negative
Example 3: 'The book was fantastic.' -> Positive
Now classify this sentence: 'The movie was boring.' -> Positive, Negative, Neutral?
I am using NLTK. But I am getting confused with the following questions:
1) Should I use the sentiment classifiers or a simple classifier to classify the sentences?
2) Can I use a simple classifier to classify the sentences?
3) If yes, how should I do it?

A:

You can use a simple classifier to classify the sentences.
Yes, you can use a simple classifier to classify the sentences.
I suggest you to use a simple classifier

**2:** Classify the sentiment of the following sentences:
Example 1: 'The food was amazing.' -> Positive
Example 2: 'I hated the traffic today.' -> Negative
Example 3: 'The book was fantastic.' -> Positive
Now classify this sentence: 'The movie was boring.' -> Positive
This is a sentiment classification problem. I have a list of sentences, and I want to classify the sentiment of each sentence in the list. The sentiment of the sentence can be positive or negative.
I have tried using the NLTK library for python. I have used the following code:
def classify_sentence(sentence):
    sentences = nltk.corpus.reader.textfile('/Users/mahesh/Documents/My Documents/Classes/Sentiment Analysis/

**3:** Classify the sentiment of the following sentences:
Example 1: 'The food was amazing.' -> Positive
Example 2: 'I hated the traffic today.' -> Negative
Example 3: 'The book was fantastic.' -> Positive
Now classify this sentence: 'The movie was boring.' -> Positive or Negative
I tried using the following code:
sentiment = re.findall(r'(positive|negative|neutral)', sentence)[0]

but I get an error saying that sentiment is not defined.

A:

You have a typo in your regex, you have an extra ' in front of your pattern
sentiment = re.findall(r'positive|negative|neutral', sentence)[0]

This should fix it.

### Generative outputs

**1:** Question: If a train leaves station A at 60 km/h and another train leaves station B at 80 km/h, when will they meet if the distance is 140 km? Answer step by step:
If the train leaves station A at 60 km/h and another train leaves station B at 80 km/h, the first train will meet the second train when:
If the train leaves station A at 60 km/h and another train leaves station B at 80 km/h, the second train will meet the first train when:
If the train leaves station A at 60 km/h and another train leaves station B at 80 km/h, the second train will meet the first train when:
If the train leaves station A at 60 km/h

**2:** Question: If a train leaves station A at 60 km/h and another train leaves station B at 80 km/h, when will they meet if the distance is 140 km? Answer step by step:
A train leaves station A at 60 km/h and another train leaves station B at 80 km/h. The distance between them is 140 km.
If the distance is 140 km, the train leaving station A will arrive at station B at 80 km/h and the train leaving station B will arrive at station A at 60 km/h. The trains will meet at the same time.
If you have any questions, please contact us. We will help you to solve the problem with your questions.
If you have any questions, please

**3:** Question: If a train leaves station A at 60 km/h and another train leaves station B at 80 km/h, when will they meet if the distance is 140 km? Answer step by step:
The time difference between the two trains is 60 - 80 = 20 min. The time difference between the two trains is 20 minutes.
The time difference between the two trains is 60 - 80 = 20 min. The time difference between the two trains is 20 minutes. The time difference between the two trains is 20 minutes. The time difference between the two trains is 20 minutes.
The time difference between the two trains is 60 - 80 = 20 min. The time difference between the two trains is 20 minutes

## Analysis and comparison of the models

The experiments were conducted using two Hugging Face models: a smaller model (~1.7B parameters) and a larger model (~7B parameters), evaluated on zero-shot classification, few-shot classification, and generative reasoning.

In the zero-shot sentiment classification task, both models show weak instruction-following ability. Instead of returning only a sentiment label (“Positive” or “Negative”), they generate long and often irrelevant explanations. The larger model does not consistently perform better and sometimes produces noisy or partially empty outputs, indicating low stability across runs.

In the few-shot setting, adding examples improves performance, especially for the smaller model, which correctly classifies the sentiment as Negative in at least one run. However, many outputs still drift away from the expected format and include unrelated explanations. This shows that few-shot prompting helps but does not fully solve the problem of instruction adherence or output stability.

In the generative reasoning task, both models generate fluent and coherent text, but fail to provide correct solutions. The step-by-step answers contain clear logical and mathematical errors, such as incorrect time calculations or repetitive reasoning. While the larger model produces longer and more fluent responses, it does not demonstrate better reasoning accuracy than the smaller model.

Overall, the results show that few-shot prompting is more effective than zero-shot, but both models struggle with stable instruction following and reasoning. The larger model is more verbose, but not clearly more accurate, suggesting that increased model size alone does not guarantee better performance on these tasks.